In [5]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('logs/ravdess-speech_attribute_SAIL-predictions.csv')

#Drop all entries that start with the name 03-01-01, since they're neutral and have no intensity variation.
df = df[~df['Filename'].str.startswith('03-01-01')]

#Add a new column with emotion category based on the filename
def get_emotion_category(filename):
    emotion_code = filename.split('-')[2]
    emotion_map = {
        '01': 'neutral',
        '02': 'calm',
        '03': 'happy',
        '04': 'sad',
        '05': 'angry',
        '06': 'fearful',
        '07': 'disgust',
        '08': 'surprised'
    }
    return emotion_map.get(emotion_code, 'unknown')

def get_emotion_intensity(filename):
    intensity_code = filename.split('-')[3]
    intensity_map = {
        '01': 'normal',
        '02': 'strong'
    }
    return intensity_map.get(intensity_code, 'unknown')

df['EmoClass'] = df['Filename'].apply(get_emotion_category)
df['Intensity'] = df['Filename'].apply(get_emotion_intensity)
df['SpkrID'] = df['Filename'].apply(lambda x: x.split('-')[6].split('.')[0])
df['Take'] = df['Filename'].apply(lambda x: x.split('-')[5])
df['Sentence'] = df['Filename'].apply(lambda x: x.split('-')[4])

# Rename columns to match MSP-Podcast format
df.rename(columns={'Arousal': 'EmoAct'}, inplace=True)
df.rename(columns={'Valence': 'EmoVal'}, inplace=True)
df.rename(columns={'Dominance': 'EmoDom'}, inplace=True)
df.rename(columns={'FileName': 'EmoDom'}, inplace=True)

#Reorder rows by speaker, then category, then intensity, then repetition number
df = df.sort_values(by=['SpkrID', 'EmoClass', 'Sentence', 'Intensity', 'Take'])

df.to_csv('logs/ravdess_attribute_differences.csv', index=False)


In [17]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('logs/ravdess_attribute_differences.csv')

# Calculate differences between strong and normal intensity for each speaker in each emotion and sentence.

# Ensure data types are correct for merging
df['SpkrID'] = df['SpkrID'].astype(str)
df['Sentence'] = df['Sentence'].astype(str)
df['Take'] = df['Take'].astype(str)

# Separate the dataframe into normal and strong intensities
normal_df = df[df['Intensity'] == 'normal'].copy()
strong_df = df[df['Intensity'] == 'strong'].copy()

# Define the columns that uniquely identify an utterance (excluding intensity and take)
id_cols = ['SpkrID', 'EmoClass', 'Sentence']

# Merge the normal takes with the first strong take
# We add suffixes to distinguish the columns from normal and strong dataframes
merged_df1 = pd.merge(
    normal_df,
    strong_df[strong_df['Take'] == '1'],
    on=id_cols,
    suffixes=('_normal', '_strong')
)

# Merge the normal takes with the second strong take
merged_df2 = pd.merge(
    normal_df,
    strong_df[strong_df['Take'] == '2'],
    on=id_cols,
    suffixes=('_normal', '_strong')
)

# Function to calculate the differences
def calculate_diffs(df, take_num):
    diff_df = df[id_cols].copy()
    diff_df['Take_strong'] = take_num
    diff_df['Act_diff'] = df['EmoAct_strong'] - df['EmoAct_normal']
    diff_df['Val_diff'] = df['EmoVal_strong'] - df['EmoVal_normal']
    diff_df['Dom_diff'] = df['EmoDom_strong'] - df['EmoDom_normal']
    return diff_df

# Calculate differences for both sets of merges
diffs1 = calculate_diffs(merged_df1, 1)
diffs2 = calculate_diffs(merged_df2, 2)

# Combine the two difference dataframes into one final result
final_diffs = pd.concat([diffs1, diffs2], ignore_index=True)

# Sort the results for better readability
final_diffs = final_diffs.sort_values(by=id_cols + ['Take_strong'])

# Display the first few rows of the result
print(final_diffs.head())

# Calculate the mean of the differences for each emotion category
mean_diffs_by_emotion = final_diffs.groupby('EmoClass')[['Act_diff', 'Val_diff', 'Dom_diff']].mean()

# Calculate mean differences by attribute
mean_diffs = final_diffs[['Act_diff', 'Val_diff', 'Dom_diff']].mean()

# Calculate mean differences by attribute excluding calm
mean_diffs_excl_calm = final_diffs[final_diffs['EmoClass'] != 'calm'][['Act_diff', 'Val_diff', 'Dom_diff']].mean()

# Display the mean differences
print("\nMean differences by emotion:")
print(mean_diffs_by_emotion)
print("\nOverall mean differences:")
print(mean_diffs)
print("\nOverall mean differences excluding calm:")
print(mean_diffs_excl_calm)

# Optionally, save the results to a new CSV
final_diffs.to_csv('logs/ravdess_intensity_differences.csv', index=False)
mean_diffs_by_emotion.to_csv('logs/ravdess_mean_intensity_differences_by_emotion.csv')
mean_diffs.to_csv('logs/ravdess_mean_intensity_differences_overall.csv')
mean_diffs_excl_calm.to_csv('logs/ravdess_mean_intensity_differences_excl_calm.csv')


    SpkrID EmoClass Sentence  Take_strong  Act_diff  Val_diff  Dom_diff
0        1    angry        1            1    0.2207   -0.0872    0.1384
1        1    angry        1            1    0.2538   -0.0751    0.1703
672      1    angry        1            2    0.1526   -0.0775    0.0889
673      1    angry        1            2    0.1857   -0.0654    0.1208
2        1    angry        2            1    0.3553   -0.1196    0.2952

Mean differences by emotion:
           Act_diff  Val_diff  Dom_diff
EmoClass                               
angry      0.265111 -0.058169  0.205155
calm      -0.027076 -0.031191 -0.021319
disgust    0.111136 -0.014257  0.090898
fearful    0.255481  0.031850  0.189343
happy      0.202205  0.020048  0.152715
sad        0.145826  0.052599  0.076151
surprised  0.101578 -0.017106  0.083978

Overall mean differences:
Act_diff    0.150609
Val_diff   -0.002318
Dom_diff    0.110989
dtype: float64

Overall mean differences excluding calm:
Act_diff    0.180223
Val_diff  